# Ask questions

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from app.core.db import connect
from app.rag.search import search
from app.llm.client import check_ollama

print("Ollama running:", check_ollama())
print(connect().execute("SELECT source_type, count(*) AS chunks FROM chunks GROUP BY 1").fetchall())

def ask(question, region=None, k=4, min_score=-1.0, source_type=None):
    print(f"Q: {question}   (region: {region or 'all'})\n")
    hits = search(question, k=k, min_score=min_score, region=region, source_type=source_type)
    if not hits:
        print("No chunk is close enough.")
    for hit in hits:
        print(f"{hit.score:.3f}  [{hit.metadata['chunk_id']}]")
        print("       " + " ".join(hit.text.split())[:220] + "...\n")

## A question about the law

In [ ]:
ask("What does the law say about forcing a child into marriage?")

## A question about cases in one region

In [ ]:
ask("Is any child being kept from school to work?", region="northern")

## Only the policies

In [ ]:
ask("Can a child be beaten as punishment at school?", source_type="policy")

In [ ]:
ask("How many high risk cases are there? Can you give me a nuber?", region="upper_east")

# Ask the assistant

The assistant chooses a tool (count, list or show cases, find by name, search the law), Python runs it,
and the model answers from the result. Numbers and verdicts come from the database, never from the model.

`region` is the team leader's region, for example `"northern"`, `"upper_east"`, `"oti"`: answers only ever
cover that region.

In [ ]:
from app.tools.assistant import ask as assistant

def ask_assistant(question, region):
    reply = assistant(question, region)
    print(f"Q: {question}   (region: {region})\n")
    print(reply.with_sources())
    print("\ntools used:", [(c["tool"], c["args"]) for c in reply.tool_calls])
    if reply.invented_citations:
        print("invented citations:", reply.invented_citations)
    return reply

## Examples

In [ ]:
ask_assistant("How many high risk cases are there? Give me a number.", region="upper_east")

In [ ]:
ask_assistant("Give me the high risk cases in Gumani.", region="northern")

In [ ]:
ask_assistant("Why was NR-SGN-SAG-0001-2025 considered high risk?", region="northern")

In [ ]:
ask_assistant("What does the law say about forcing a child into marriage?", region="northern")

## Finding a case by name

The first cell asks for the OK; the second runs the search only after it.

In [ ]:
reply = ask_assistant("Do we have a case for Ama Mensah?", region="northern")

In [ ]:
reply = assistant("Do we have a case for Ama Mensah?", "northern", approved=reply.needs_approval)
print(reply.with_sources())

In [ ]:
ask_assistant("which cases are urgent?", region="northern")

In [ ]:
ask_assistant("which cases had the urgent indicators?", region="northern")